# Bước 1 — Khám phá dữ liệu TMDB 5000

**Mục tiêu:** đọc 2 file CSV gốc và trình bày rõ **một dòng có những cột nào**, và các cột JSON khi parse ra có **những trường nào**.

Nguồn: `data/raw/tmdb_5000_movies.csv` và `data/raw/tmdb_5000_credits.csv`.

## 1. Đọc dữ liệu

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)

# chạy được cả khi mở notebook từ thư mục gốc lẫn từ thư mục notebook/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
RAW = ROOT / "data" / "raw"

movies = pd.read_csv(RAW / "tmdb_5000_movies.csv")
credits = pd.read_csv(RAW / "tmdb_5000_credits.csv")
print("movies :", movies.shape)
print("credits:", credits.shape)

movies : (4803, 20)
credits: (4803, 4)


In [2]:
movies.head(3)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adven...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""na...",en,Avatar,"In the 22nd century, a paraplegic Marine is dispatched t...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289}, {""name""...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fa...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come bac...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""name"": ""Je...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adven...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name"": ""based ...",en,Spectre,A cryptic message from Bond’s past sends him on a trail ...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""name"": ""Danja...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}, {""iso_3...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""}, {""iso_639...",Released,A Plan No One Escapes,Spectre,6.3,4466


In [3]:
credits.head(3)

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""credit_id""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""department"":..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Sparrow"", ""cr...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""department"":..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""credit_id"": ...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""department"":..."


## 2. Kiểm tra nhanh: kiểu dữ liệu, thiếu, trùng, khoá nối

In [4]:
info = lambda df: pd.DataFrame({"dtype": df.dtypes, "null": df.isna().sum(), "nunique": df.nunique(),
                                   "ví dụ": df.iloc[0].astype(str).str[:50]})
info(movies)

,dtype,null,nunique,ví dụ
budget,int64,0,436,237000000
genres,object,0,1175,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"":"
homepage,object,3091,1691,http://www.avatarmovie.com/
id,int64,0,4803,19995
keywords,object,0,4222,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 296"
original_language,object,0,37,en
original_title,object,0,4801,Avatar
overview,object,3,4800,"In the 22nd century, a paraplegic Marine is dispat"
popularity,float64,0,4802,150.437577
production_companies,object,0,3697,"[{""name"": ""Ingenious Film Partners"", ""id"": 289}, {"


In [5]:
info(credits)

,dtype,null,nunique,ví dụ
movie_id,int64,0,4803,19995
title,object,0,4800,Avatar
cast,object,0,4761,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""cred"
crew,object,0,4776,"[{""credit_id"": ""52fe48009251416c750aca23"", ""depart"


In [6]:
print("movies.id duy nhất       :", movies["id"].is_unique)
print("credits.movie_id duy nhất:", credits["movie_id"].is_unique)
print("2 file cùng tập movie id :", set(movies["id"]) == set(credits["movie_id"]))
print("title trùng nhau         :", movies["title"].duplicated().sum(), "(title không phải khoá -> dùng id làm URI)")

movies.id duy nhất       : True
credits.movie_id duy nhất: True
2 file cùng tập movie id : True
title trùng nhau         : 3 (title không phải khoá -> dùng id làm URI)


## 3. Parse các cột JSON

7 cột sau là **chuỗi JSON** nằm trong ô CSV. Sau khi parse, mỗi ô là một list các dict, mỗi dict là một phần tử (một thể loại, một diễn viên, một thành viên crew...).

In [7]:
JSON_COLS = {
    "movies": ["genres", "keywords", "production_companies", "production_countries", "spoken_languages"],
    "credits": ["cast", "crew"],
}

def parse(cell):
    return json.loads(cell) if isinstance(cell, str) and cell else []

for col in JSON_COLS["movies"]:
    movies[col] = movies[col].map(parse)
for col in JSON_COLS["credits"]:
    credits[col] = credits[col].map(parse)

Tách mỗi cột JSON thành một bảng: 1 dòng = 1 phần tử, kèm `movie_id` để biết phần tử thuộc phim nào.

In [8]:
def explode(df, key, col):
    """Biến cột list-of-dict thành bảng phẳng, có cột movie_id."""
    rows = df[[key, col]].explode(col).dropna(subset=[col])
    out = pd.json_normalize(rows[col].tolist())
    out.insert(0, "movie_id", rows[key].values)
    return out

tables = {col: explode(movies, "id", col) for col in JSON_COLS["movies"]}
tables |= {col: explode(credits, "movie_id", col) for col in JSON_COLS["credits"]}

### Các trường có trong mỗi cột JSON

In [9]:
rows = [(name, c, t[c].dtype, t[c].isna().sum(), str(t[c].iloc[0])[:40])
        for name, t in tables.items() for c in t.columns.drop("movie_id")]
pd.DataFrame(rows, columns=["cột JSON", "trường", "kiểu", "null", "ví dụ"]).set_index(["cột JSON", "trường"])

kiểu  null                     ví dụ
cột JSON             trường                                            
genres               id           int64     0                        28
                     name        object     0                    Action
keywords             id           int64     0                      1463
                     name        object     0             culture clash
production_companies name        object     0   Ingenious Film Partners
                     id           int64     0                       289
production_countries iso_3166_1  object     0                        US
                     name        object     0  United States of America
spoken_languages     iso_639_1   object     0                        en
                     name        object     0                   English
cast                 cast_id      int64     0                       242
                     character   object     0                Jake Sully
                     credit_id   object     0  5602a8a7c3a3685532001c9a
                     gender       int64     0                         2
                     id           int64     0                     65731
                     name        object     0           Sam Worthington
                     order        int64     0                         0
crew                 credit_id   object     0  52fe48009251416c750aca23
                     department  object     0                   Editing
                     gender       int64     0                         0
                     id           int64     0                      1721
                     job         object     0                    Editor
                     name        object     0         Stephen E. Rivkin

## 4. Bảng tổng hợp: tất cả các trường

Các trường của cột JSON (đã parse) ở trên, nối tiếp phía dưới là các trường không phải JSON của 2 file.

In [10]:
parent = {c: f for f, cols in JSON_COLS.items() for c in cols}
val = lambda s: (s.dtype, s.isna().sum(), str(s.iloc[0])[:40])

json_rows = [("JSON", f"{parent[n]}.{n}", c, *val(t[c])) for n, t in tables.items() for c in t.columns.drop("movie_id")]
plain_rows = [("Không phải JSON", f, c, *val(df[c])) for f, df in [("movies", movies), ("credits", credits)]
              for c in df.columns if c not in JSON_COLS[f]]

pd.DataFrame(json_rows + plain_rows, columns=["loại", "nguồn", "trường", "kiểu", "null", "ví dụ"]).set_index(["loại", "nguồn", "trường"])

kiểu  null                                     ví dụ
loại            nguồn                       trường                                                                    
JSON            movies.genres               id                   int64     0                                        28
                                            name                object     0                                    Action
                movies.keywords             id                   int64     0                                      1463
                                            name                object     0                             culture clash
                movies.production_companies name                object     0                   Ingenious Film Partners
                                            id                   int64     0                                       289
                movies.production_countries iso_3166_1          object     0                                        US
                                            name                object     0                  United States of America
                movies.spoken_languages     iso_639_1           object     0                                        en
                                            name                object     0                                   English
                credits.cast                cast_id              int64     0                                       242
                                            character           object     0                                Jake Sully
                                            credit_id           object     0                  5602a8a7c3a3685532001c9a
                                            gender               int64     0                                         2
                                            id                   int64     0                                     65731
                                            name                object     0                           Sam Worthington
                                            order                int64     0                                         0
                credits.crew                credit_id           object     0                  52fe48009251416c750aca23
                                            department          object     0                                   Editing
                                            gender               int64     0                                         0
                                            id                   int64     0                                      1721
                                            job                 object     0                                    Editor
                                            name                object     0                         Stephen E. Rivkin
Không phải JSON movies                      budget               int64     0                                 237000000
                                            homepage            object  3091               http://www.avatarmovie.com/
                                            id                   int64     0                                     19995
                                            original_language   object     0                                        en
                                            original_title      object     0                                    Avatar
                                            overview            object     3  In the 22nd century, a paraplegic Marine
                                            popularity         float64     0                                150.437577
                                            release_date        object     1                                2009-12-10
                                            revenue              int64     0                                2787965087
                    